In [ ]:
import sklearn

print(f"Scikit-Learn Version: {sklearn.__version__}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('bank_final.csv')

# Check rows and columns
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

# Check for missing data
print("\nMissing Data Count per Column:")
print(df.isnull().sum())

In [ ]:
# Scatterplot: Age vs Balance
plt.figure(figsize=(8, 5))
plt.scatter(df['Age'], df['Balance'], alpha=0.5, color='steelblue')
plt.title('Scatterplot: Age vs Balance')
plt.xlabel('Age')
plt.ylabel('Balance')
plt.show()

# Density Chart: Balance
plt.figure(figsize=(8, 5))
df['Balance'].plot(kind='density', color='coral', linewidth=2)
plt.title('Density Chart: Account Balance')
plt.xlabel('Balance')
plt.xlim(-5000, df['Balance'].max()) # Adjusting x-limit to handle long tails
plt.show()

In [ ]:
# Correlation matrix (numeric columns only)
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numeric Features')
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Separate the target variable before creating dummy variables
# Assuming 'Signed Up' is coded as 'yes'/'no'. If it's already 1/0, this mapping will safely ignore it.
target = df['Signed Up'].map({'yes': 1, 'no': 0}) if df['Signed Up'].dtype == 'object' else df['Signed Up']

# Drop the target from features
features = df.drop('Signed Up', axis=1)

# Create dummy variables for categorical columns
features_encoded = pd.get_dummies(features, drop_first=True)
print(f"Total columns after creating dummy variables: {features_encoded.shape[1]}")

# Normalize the numeric data (crucial for distance-based kNN)
scaler = MinMaxScaler()
features_scaled = pd.DataFrame(scaler.fit_transform(features_encoded), columns=features_encoded.columns)

In [ ]:
from sklearn.model_selection import train_test_split

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    features_scaled, target, test_size=0.2, random_state=42
)
print(f"Training set size: {X_train.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Testing 10 different k-values (odd numbers from 1 to 19)
k_values = range(1, 20, 2)
accuracies = []

print("--- k-Value Accuracies ---")
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print(f"k = {k:2d} | Accuracy: {acc:.4f}")

# Plot the accuracies to make it easy to see the peak
plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, marker='o', linestyle='dashed', color='green')
plt.title('kNN Accuracy vs. k Value')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Accuracy Score')
plt.xticks(k_values)
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Retrain the model with the best k-value you found (replace '7' if your graph peaks elsewhere)
best_k = 7
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train, y_train)
final_predictions = final_knn.predict(X_test)

# Generate and plot the Confusion Matrix
cm = confusion_matrix(y_test, final_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Did Not Sign Up', 'Signed Up'])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap='Blues', ax=ax)
plt.title(f"Confusion Matrix (k={best_k})")
plt.show()

In [ ]:
import fairlearn.metrics

# To evaluate fairness, we need the original, un-dummied sensitive feature aligned with the test set.
# Assuming 'Gender' or 'Race' are the protected classes. 
sensitive_feature = features.loc[X_test.index, 'Gender'] # Change 'Gender' to 'Race' to test that instead

# Calculate Demographic Parity and Equalized Odds
dpr = fairlearn.metrics.demographic_parity_ratio(
    y_test, final_predictions, sensitive_features=sensitive_feature
)
eor = fairlearn.metrics.equalized_odds_ratio(
    y_test, final_predictions, sensitive_features=sensitive_feature
)

print("--- Fairness Metrics ---")
print(f"Protected Class: Gender")
print(f"Demographic Parity Ratio (DPR): {dpr:.4f}")
print(f"Equalized Odds Ratio (EOR):     {eor:.4f}")